In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Baiesi-Paczuski Earthquake Network — Italy INGV Catalog (1985-2025)

Run as a script  : python ITALY_network_BP.py
Convert to notebook: python convert_to_notebook.py ITALY_network_BP.py notebooks/ITALY_network_BP.ipynb
"""

import logging
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns

from src.network import build_baiesi_paczuski_network
from src.metrics import estimate_gamma_mle, test_power_law, measure_pa_forest, plot_preferential_attachment
from src.assortativity import compute_assortativity, plot_assortativity, plot_knn, plot_rich_club
from src.centrality import (
    plot_centrality_correlation,
    plot_top_n_cells,
    plot_geo_top_n_interactive,
    plot_geo_centrality_overlap,
    compute_bb_fitness_events,
    plot_bb_fitness,
    plot_bb_fitness_theory,
    plot_bb_fitness_geo,
)
from src.community import (
    run_louvain,
    run_consensus_louvain,
    run_spectral,
    run_infomap,
    run_directed_louvain,
    run_hdbscan_spectral,
    run_hdbscan_geo,
    run_bigclam,
    compare_partitions,
    plot_partition_scores,
    compute_nmi_matrix,
    plot_nmi_heatmap,
    plot_community_geo,
)
from src.viz import (
    analyze_degree_distribution,
    analyze_degree_distribution_log_binning,
    analyze_in_out_degree_distribution,
    plot_ccdf_with_fit,
    plot_degree_distribution_linear,
)
from src.plotutils import setup_matplotlib, configure_saves, savefig, save_plotly

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

DATA_DIR    = Path("data/INGV")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data").mkdir(exist_ok=True)
(RESULTS_DIR / "cache").mkdir(exist_ok=True)
(RESULTS_DIR / "gephi").mkdir(exist_ok=True)

CUT_YEAR   = 1985
TARGET_CRS = "epsg:32632"  # UTM Zone 32N — Italy

# BP model hyperparameters (Baiesi & Paczuski 2004, PRE 69, 066106)
BP_B          = 1.0    # Gutenberg-Richter b-value
BP_ALPHA      = 1.0    # Omori time exponent
BP_D          = 1.6    # fractal dimension of epicentre distribution
BP_T_MAX_DAYS = 730.0  # look-back window (2 years)

SAVE_PDF: bool = True
SAVE_JPG: bool = True

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "italy" / "bp")

## Data Loading

The INGV catalog covers Italian seismicity 1985–2025 at M ≥ 1.5.  Events are
filtered to `year ≥ 1985` (catalog completeness threshold) and sorted by UTC
origin time.  The reset index becomes the integer node ID used throughout the BP
and ZBZ analyses.

**Catalog properties:** ≈ 215,000 events; latitude 34°–48° N, longitude 3°–22° E
(includes Ionian islands and Adriatic Sea as well as the peninsula);
depth range −5 to 650 km (negative values are surface-rounding artefacts, kept as-is);
magnitude range 1.5–6.9 (Italy has no M≥7 since 1980 Irpinia in this catalog).

**Expected output:** ≈ 215,000 rows printed, RAM ≈ 30–60 MB.  The head/tail
table confirms chronological ordering from January 1985 to late 2025.

In [ ]:
print("Loading Italy earthquake catalog...")
df = pd.read_csv(DATA_DIR / "italy_earthquakes_1985_2025.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df_net = (
    df[df["time"].dt.year >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)

print(f"Loaded {len(df_net):,} earthquakes "
      f"({df_net['time'].dt.year.min()}–{df_net['time'].dt.year.max()})")
print(f"RAM: {df_net.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
display(pd.concat([df_net.head(3), df_net.tail(3)]))

## Network Construction

The **Baiesi-Paczuski (2004)** model assigns a directed link $i \to j$ when
earthquake $i$ is the most likely "parent" of $j$ — the past event minimising
the spatio-temporal proximity metric:

$$n_{ij} = t_{ij}^{\alpha} \cdot r_{ij}^{d} \cdot 10^{-b\, m_i}$$

where $t_{ij}$ = time separation (days), $r_{ij}$ = epicentral distance (km),
$m_i$ = parent magnitude, and the standard parameters are $b = 1.0$ (empirical
Gutenberg-Richter value for Italy), $\alpha = 1.0$ (Omori-law exponent),
$d = 1.6$ (fractal dimension of Italian seismicity).  Only events within the
$t_{\max} = 730$-day look-back window are candidates.

Each event has **at most one parent**, producing a **directed forest** whose
trees correspond to aftershock families.  Nodes are individual earthquake events
(not spatial cells as in ABE), so every link is a genuine seismological
parent–child relationship — not a geographic co-occurrence.

*Reference:* Baiesi, M., & Paczuski, M. (2004). Scale-free networks of
earthquakes and aftershocks. *Physical Review E*, 69(6), 066106.

**Expected output:** ≈ 215,000 nodes and ≈ 214,999 edges (N − 1 for a single
spanning tree over a dense continuous catalog with no 730-day gaps).
Build time ≈ 5–10 minutes; the INFO logger prints progress at 5% intervals.

In [ ]:
print(f"Building BP network (b={BP_B}, α={BP_ALPHA}, d={BP_D}, "
      f"t_max={BP_T_MAX_DAYS:.0f} days)…")
t_build = time.time()
G = build_baiesi_paczuski_network(
    df_net,
    b=BP_B,
    alpha=BP_ALPHA,
    d=BP_D,
    t_max_days=BP_T_MAX_DAYS,
)
print(f"Build time: {time.time() - t_build:.1f}s")
print(f"Nodes: {G.number_of_nodes():,}  Edges: {G.number_of_edges():,}")

## Tree Structure Verification

By construction every non-root node in the BP forest has exactly one incoming
edge (its unique parent).  Three structural checks are performed:

1. **Max in-degree ≤ 1** — any value $> 1$ indicates a bug in the parent
selection loop.
2. **Roots** (in-degree = 0) — events for which no prior event exists within
the $t_{\max}$ look-back window.  For a dense continuous catalog the only
root is the chronologically first event in the catalog.
3. **Leaves** (out-degree = 0) — events that never acted as a parent; the vast
majority of the catalog (most earthquakes are isolated or too small to be
selected as a parent).

For the Italy INGV catalog 1985–2025 (M ≥ 1.5, several events per day on
average), there is no 730-day gap in seismicity, so every event except the
chronologically first one finds a parent.  The BP algorithm therefore produces
a **single spanning tree**: 1 root, 1 weakly connected component.  This is the
correct and expected result — not a defect.  The tree structure still encodes
meaningful information: high out-degree nodes are mainshocks that directly
triggered many aftershocks, and node depth (hops from root) measures cascade length.

**Expected output:** max in-degree = 1, roots = 1 (0.0%), trees = 1.
Top-10 tree sizes: [≈215000] — a single tree spanning the full catalog.

In [ ]:
in_degrees  = [d for _, d in G.in_degree()]
out_degrees = [d for _, d in G.out_degree()]

max_in  = max(in_degrees)
n_roots = sum(1 for d in in_degrees  if d == 0)
n_leaves= sum(1 for d in out_degrees if d == 0)
n_trees = nx.number_weakly_connected_components(G)

print(f"Max in-degree : {max_in}  (must be ≤ 1 for a valid forest)")
print(f"Roots (in=0)  : {n_roots:,}  ({100*n_roots/G.number_of_nodes():.1f}%)")
print(f"Leaves (out=0): {n_leaves:,}  ({100*n_leaves/G.number_of_nodes():.1f}%)")
print(f"Trees (WCCs)  : {n_trees:,}")

# Size distribution of the 10 largest trees
wcc_sizes = sorted(
    [len(c) for c in nx.weakly_connected_components(G)], reverse=True
)
print("\nTop-10 tree sizes:", wcc_sizes[:10])

## Hub Map — 2D

The top 100 events by out-degree are plotted on a geographic map, coloured and
sized by out-degree (direct number of events each earthquake triggered as their
nearest BP parent).  Marker size is linearly scaled to [8, 35] pixels; points
are rendered in ascending out-degree order so the highest values appear on top.

In the BP model, **high out-degree ↔ productive mainshock**: events that
closely preceded many smaller earthquakes in time and space, minimising $n_{ij}$
for a large cluster of followers.  A fixed top-100 is used (not 2%) because
with ≈ 215,000 event-level nodes, 2% would plot ≈ 4,300 overlapping points.

**Expected output:** hubs cluster along the Apennine arc (Central Italy),
with secondary concentrations in Calabria/Sicily and Friuli.  The highest
out-degree events should correspond to documented major sequences: L'Aquila
2009 (Mw 6.1), Amatrice–Norcia 2016 (Mw 6.2–6.5), Irpinia 1980 (Mw 6.9).
Hover on each point to see event index and magnitude.

In [ ]:
df_nodes = pd.DataFrame([
    {
        "event_idx":  n,
        "out_degree": G.out_degree(n),
        "lat":        G.nodes[n]["lat"],
        "lon":        G.nodes[n]["lon"],
        "magnitude":  G.nodes[n]["magnitude"],
    }
    for n in G.nodes()
])
df_hubs = df_nodes.nlargest(100, "out_degree").copy()
print(f"Top 100 hubs: out_degree range [{df_hubs['out_degree'].min():.0f}, {df_hubs['out_degree'].max():.0f}]")

# Sort ascending so high out_degree markers render on top in plotly
df_hubs = df_hubs.sort_values("out_degree")

# Scale size to [8, 35] range
_od_min = df_hubs["out_degree"].min()
_od_max = df_hubs["out_degree"].max()
df_hubs["marker_size"] = 8 + 27 * (df_hubs["out_degree"] - _od_min) / max(_od_max - _od_min, 1)

_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

fig = px.scatter_map(
    df_hubs, lat="lat", lon="lon",
    color="out_degree", size="marker_size",
    size_max=35, opacity=0.85,
    color_continuous_scale="inferno",
    map_style="carto-positron",
    hover_name="event_idx",
    hover_data={"magnitude": ":.2f", "out_degree": True, "marker_size": False},
    title="BP Network Hubs: Top 100 Most Productive Events — Italy",
)
fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    width=MAP_WIDTH, height=MAP_HEIGHT,
    map=dict(center={"lat": 41.9, "lon": 12.5}, zoom=0, bounds=_IT_BOUNDS),
)
save_plotly(fig, "bp_hub_map_2d_italy")
fig.show()

## Out-Degree Distribution — Linear

Linear-scale bar chart of out-degree (number of direct aftershocks per event),
truncated at $k = 50$ and plotted on a log y-axis.

Since BP in-degree $\in \{0, 1\}$ for all nodes, total degree $\approx$
out-degree; the functions below act on total degree but the difference is at
most 1 per node.

**Expected output:** a sharply declining distribution — the vast majority of
events have out-degree 0 (leaves) or 1 (pass-through nodes in the chain).
The bar at $k = 0$ dominates the y-axis.  The log scale exposes the long tail
extending to $k \approx 100$–500 for the most productive mainshocks.

In [ ]:
plot_degree_distribution_linear(G, "BP Italy", max_degree=50, weighted=False)

## Out-Degree Distribution — Log-Log

Log-log scatter of the probability mass $P(k)$ vs degree $k$ for in-degree and
out-degree separately.

For the BP forest:
- **In-degree** is trivially $\{0, 1\}$ — two points only, not power-law.
- **Out-degree** follows $P(k_{\text{out}}) \propto k_{\text{out}}^{-\gamma}$
(Baiesi & Paczuski 2004, Fig. 2), reflecting the Omori-law clustering of
aftershocks: most events trigger zero children while rare mainshocks trigger
hundreds.  The power-law exponent $\gamma \approx 2$–$2.5$ is related to
the Gutenberg-Richter b-value and the fractal dimension $d$.

**Expected output:** in-degree scatter — two isolated points; out-degree scatter
— approximately linear on the log-log plot for $k \geq 2$, with visible scatter
at the high-$k$ tail (finite-size noise from ≈ 5–20 events in each large-$k$ bin).

In [ ]:
analyze_in_out_degree_distribution(G, "BP Italy")
analyze_degree_distribution(G, "BP Italy")

## Out-Degree Distribution — Log Binning

Logarithmic binning (Clauset et al. 2009) reduces noise in the sparse high-$k$
tail by pooling observations into geometrically spaced bins and normalising by
bin width to yield probability *density* $P(k)$ rather than probability *mass*.

The fitted slope gives a visual estimate of $\gamma_{\text{out}}$ before the
more rigorous MLE step.  Log-binning is the preferred method for verifying
power-law tails when the distribution spans several decades of $k$.

**Expected output:** a roughly linear log-binned curve for $k \geq 2$ with
slope ≈ −2 to −2.5.  Deviations at very high $k$ (rightmost bins) are normal
finite-sample noise.  The fit quality (R²) should exceed 0.85.

In [ ]:
analyze_degree_distribution_log_binning(G, "BP Italy", k_min_fit=2)

## Out-Degree Distribution — CCDF

The complementary CDF $P(K \geq k)$ is plotted on a log-log scale.  Unlike
the probability mass, the CCDF is monotone non-increasing and avoids binning
artefacts entirely.  For a power-law $P(k) \propto k^{-\gamma}$, the CCDF
has slope $-(\gamma - 1)$ on the log-log plot.

**Expected output:** an approximately linear CCDF for $k \geq 2$ with slope
≈ −1 to −1.5 (corresponding to $\gamma \approx 2$–$2.5$).  An exponential
distribution would appear as a concave curve on this log-log plot; a straight
line confirms the power-law tail.

In [ ]:
plot_ccdf_with_fit(G, "BP Italy", k_min_fit=2)

## MLE Gamma — Branching Exponent

Maximum-likelihood estimate of the out-degree tail exponent (Clauset et al. 2009):

$$\hat{\gamma} = 1 + n\left[\sum_{i=1}^{n}\ln\frac{k_i}{k_{\min}}\right]^{-1}$$

where the sum runs over the $n$ events with $k_{\text{out}} \geq k_{\min} = 1$.

The **branching ratio** $\sigma = \langle k_{\text{out}} \rangle$ (mean out-degree
over all nodes) controls epidemic-like growth of aftershock sequences:
$\sigma < 1$ (subcritical) — sequences die out; $\sigma = 1$ — criticality;
$\sigma > 1$ — supercritical growth.  Real seismicity is subcritical
($\sigma \approx 0.1$–$0.3$) for sparse catalogs, but can approach 1 for
dense catalogs where the BP algorithm forces every event to have a parent.

The CSN likelihood-ratio test (Clauset et al. 2009) compares the power-law
fit to an exponential alternative: $R > 0$ favours power-law; $p < 0.05$
confirms statistical significance.

*Reference:* Baiesi & Paczuski (2004) found $\gamma \approx 2.0$–$2.5$
for Southern California; Abe & Suzuki (2004) reported similar values for Japan.

**Expected output:** $\hat{\gamma} \approx 2.0$–$2.5$; $\sigma \approx 0.9$–$1.0$
(near-critical due to the dense continuous Italy catalog); CSN verdict likely
"power law" with $p \approx 0$–$0.3$.

In [ ]:
out_degs = [d for _, d in G.out_degree() if d > 0]
sigma    = np.mean([d for _, d in G.out_degree()])

gamma_bp = estimate_gamma_mle(out_degs, k_min=1)
print(f"Branching ratio σ = ⟨k_out⟩ = {sigma:.4f}  "
      f"({'subcritical — sequences die out' if sigma < 1 else 'supercritical'})")
print(f"MLE γ (out-degree, k_min=1): {gamma_bp:.3f}")

result_bp = test_power_law(out_degs, k_min=1)
print(f"  CSN test: γ={result_bp['gamma']} (σ={result_bp['sigma']})  "
      f"R={result_bp['R']}  p={result_bp['p_value']}  → {result_bp['verdict']}")

## Preferential Attachment

The empirical attachment kernel $\pi(k) \propto k^{\alpha}$ measures whether
nodes with higher current degree attract new edges at a faster rate:

$$\pi(k) = \frac{\sum_{i:\,k_i^{\text{out}}(t)=k} \Delta k_i}
{\#\{i:\,k_i^{\text{out}}(t)=k\}}$$

**For the BP forest, $\alpha \approx 0$ is the theoretically expected result.**
Parent selection minimises $\eta_{ij} = t_{ij}^{\alpha} r_{ij}^{d} 10^{-bm_i}$
— a function of inter-event time, epicentral distance, and parent magnitude.
The current out-degree of $i$ does not enter the metric, so
a node's chance of being chosen as a parent is independent of how many
children it already has.  A flat $\pi(k)$ ($\alpha \approx 0$) confirms
this: no preferential attachment in the BP forest.

**Seismological implication:** the scale-free out-degree distribution
of the BP forest does *not* arise from degree-dependent growth.
Instead, the heavy tail is generated by magnitude heterogeneity through
the $10^{-bm_i}$ factor — large-magnitude events minimise $\eta_{ij}$
for a wide spatial and temporal neighbourhood, accumulating many
children.  The power law follows from the Gutenberg–Richter exponential
magnitude distribution, not from a rich-get-richer mechanism.

*Reference:* Jeong H., Néda Z. & Barabási A.-L. (2003). Measuring
preferential attachment in evolving networks. *Europhysics Letters*,
61(4), 567–572.

In [ ]:
print("Measuring preferential attachment kernel (BP forest)…")
ks_pa, pi_k_pa, alpha_pa = measure_pa_forest(G, df_net)
plot_preferential_attachment(ks_pa, pi_k_pa, alpha_pa, title="BP Italy (forest PA)")

## Macroscopic Metrics

For a directed forest, standard small-world metrics (clustering, path length)
are computed on the **undirected giant component** — the largest weakly connected
subgraph (for Italy BP: the single spanning tree = entire catalog).

**Depth distribution:** BFS from the unique root (the chronologically first
event) gives the hop distance of every event from the origin.  Depth measures
cascade length — how many triggering steps a given event is removed from the
start of the catalog.  For a branching tree with $N$ nodes and branching ratio
$\sigma$, mean depth $\approx \log N / \log \langle k \rangle$.

**Clustering and transitivity** are exactly 0 for a pure tree (no triangles),
so any nonzero value reflects numerical precision or edge-weight ties in the
construction.

**Expected output:** giant tree = 100% of catalog (≈ 215,000 nodes).
Depth histogram: heavily right-skewed — most events are depth 1–10 (direct
children of recent larger events); a thin tail extends to depths of several
thousand (long cascades).  Avg clustering ≈ 0.000, transitivity ≈ 0.000.

In [ ]:
wcc_list   = list(nx.weakly_connected_components(G))
giant_wcc  = max(wcc_list, key=len)
G_giant    = G.subgraph(giant_wcc).copy()
G_und_giant = G_giant.to_undirected()

pct = G_giant.number_of_nodes() / G.number_of_nodes() * 100
print(f"Giant tree: {G_giant.number_of_nodes():,} nodes ({pct:.1f}% of catalog)")
print(f"Trees total: {len(wcc_list):,} | Largest 5 sizes: {sorted([len(c) for c in wcc_list], reverse=True)[:5]}")

# Depth distribution in the giant tree (BFS from root = node with in-degree 0)
_root = next(n for n in G_giant.nodes() if G_giant.in_degree(n) == 0)
_depths_bfs = nx.single_source_shortest_path_length(G_giant, _root)
_depth_arr  = np.array(list(_depths_bfs.values()))
print(f"\nGiant tree depth (from root):")
print(f"  Max depth  : {_depth_arr.max()}")
print(f"  Mean depth : {_depth_arr.mean():.2f}")
print(f"  Median depth: {np.median(_depth_arr):.0f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(_depth_arr, bins=50, color="steelblue", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Depth from sequence root (hops)", fontsize=12)
ax.set_ylabel("Events", fontsize=12)
ax.set_title("Triggering-chain depth distribution — BP Italy (giant tree)", fontsize=13)
ax.set_yscale("log")
ax.grid(axis="y", ls="--", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
savefig("bp_depth_distribution_italy")
plt.show()

# Giant component clustering and transitivity on undirected version
avg_c = nx.average_clustering(G_und_giant)
transitivity = nx.transitivity(G_und_giant)
print(f"\nGiant tree (undirected): avg clustering = {avg_c:.4f}  "
      f"| transitivity = {transitivity:.4f}")
print("(Both ≈ 0 expected for a pure tree — any nonzero value reflects edge-weight ties)")

## Centrality Analysis

## Centrality Analysis — 12 Measures

Twelve centrality measures are computed inline (the shared `compute_all_centralities`
function expects ABE-style cell-string node IDs and is not used here):

### Degree family
- **In-Degree** $k^{\mathrm{in}}_i$: in a forest, every non-root event has
$k^{\mathrm{in}} = 1$ (one parent); roots have $k^{\mathrm{in}} = 0$.
In-degree centraliy thus identifies *background events* (roots) vs *triggered events*.
- **Out-Degree** $k^{\mathrm{out}}_i$: direct aftershock offspring count — the
classical seismic *productivity* measure (Helmstetter 2003).
- **Degree** $(k^{\mathrm{in}} + k^{\mathrm{out}})$: total connectivity.

### Path-based measures
- **PageRank**: long-run influence along triggering chains — events consistently
reached from many starting points receive high PR.  Concentrates on early
events high in the tree (common ancestors of large subtrees).
- **Harmonic** $H_i = \sum_{j \neq i} 1/d_{ij}$: essential for forests since
closeness = 0 for any node in a weakly connected component different from the
query's; harmonic handles disconnected subforests gracefully.
- **Betweenness** ($k = 500$ sample): fraction of shortest paths passing through
each node — events on the main trunk (spine from root to largest subtrees) dominate.

### Spectral measures
- **Eigenvector / Katz** ($\alpha = 0.85 / k_{\max}$): neighbour-weighted importance.
On trees these converge toward out-degree; divergence identifies nodes with
influential neighbours beyond their own direct productivity.
- **HITS Hub / Authority**: Hub = sends many links (triggers productive subtrees);
Authority = receives many links (attracted many children).

### Local topology
- **Clustering coefficient**: fraction of neighbour pairs that are mutually connected
(undirected version).  Exactly 0 for a pure mathematical tree; any non-zero value
in the BP network indicates rare multi-parent links (events with two η-closest
predecessors at equal metric distance).
- **Triangle count**: raw triangles per node (undirected).  Positive triangles
confirm non-tree structure; diagnostic for BP edge-creation ties.

**Expected output:** top-5 per metric overlaps substantially with hub-map top-100.
Betweenness ≈ 5–10 min with $k = 500$.  Eigenvector may require numpy fallback.

In [ ]:
print("Computing centralities for BP network (may take a few minutes)…")
_t0_cent = time.time()

G_nsl     = G.copy()
G_nsl.remove_edges_from(nx.selfloop_edges(G_nsl))
G_und_all = G.to_undirected()
G_und_all.remove_edges_from(nx.selfloop_edges(G_und_all))
_N        = G.number_of_nodes()

in_deg_cent  = nx.in_degree_centrality(G)
out_deg_cent = nx.out_degree_centrality(G)
deg_cent     = nx.degree_centrality(G)
print(f"  Degree (in/out/total) done ({time.time()-_t0_cent:.1f}s)")

pr_cent  = nx.pagerank(G, weight="weight")
print(f"  PageRank done ({time.time()-_t0_cent:.1f}s)")

harm_cent = nx.harmonic_centrality(G)
print(f"  Harmonic done ({time.time()-_t0_cent:.1f}s)")

bet_cent = nx.betweenness_centrality(G, k=min(500, _N), seed=42, normalized=True)
print(f"  Betweenness done ({time.time()-_t0_cent:.1f}s)")

try:
    eig_cent = nx.eigenvector_centrality(G_und_all, weight="weight", max_iter=500, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    try:
        eig_cent = nx.eigenvector_centrality_numpy(G_und_all, weight="weight")
    except nx.AmbiguousSolution:
        eig_cent = {n: 0.0 for n in G.nodes()}
        print("  Eigenvector: skipped (graph is disconnected)")
print(f"  Eigenvector done ({time.time()-_t0_cent:.1f}s)")

_max_deg   = max((G.degree(n) for n in G.nodes()), default=1)
_alpha_katz = 0.85 / _max_deg
try:
    katz_cent = nx.katz_centrality(
        G, alpha=_alpha_katz, weight="weight", normalized=True, max_iter=1000, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    katz_cent = nx.katz_centrality_numpy(G, alpha=_alpha_katz, weight="weight")
print(f"  Katz done ({time.time()-_t0_cent:.1f}s)")

try:
    hits_hub, hits_auth = nx.hits(G_nsl, max_iter=1000, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    _zeros    = {n: 0.0 for n in G.nodes()}
    hits_hub  = _zeros.copy()
    hits_auth = _zeros.copy()
print(f"  HITS done ({time.time()-_t0_cent:.1f}s)")

clust_cent = nx.clustering(G_und_all, weight="weight")
print(f"  Clustering done ({time.time()-_t0_cent:.1f}s)")
tri_count = nx.triangles(G_und_all)
print(f"  Triangles done ({time.time()-_t0_cent:.1f}s total)")

df_centrality = pd.DataFrame([
    {
        "cell_id":     n,
        "lat":         G.nodes[n]["lat"],
        "lon":         G.nodes[n]["lon"],
        "depth_km":    G.nodes[n]["depth_km"],
        "In_Degree":   in_deg_cent.get(n, 0.0),
        "Out_Degree":  out_deg_cent.get(n, 0.0),
        "Degree":      deg_cent.get(n, 0.0),
        "PageRank":    pr_cent.get(n, 0.0),
        "Harmonic":    harm_cent.get(n, 0.0),
        "Betweenness": bet_cent.get(n, 0.0),
        "Eigenvector": eig_cent.get(n, 0.0),
        "Katz":        katz_cent.get(n, 0.0),
        "HITS_Hub":    hits_hub.get(n, 0.0),
        "HITS_Auth":   hits_auth.get(n, 0.0),
        "Clustering":  clust_cent.get(n, 0.0),
        "Triangles":   float(tri_count.get(n, 0)),
    }
    for n in G.nodes()
    if "lat" in G.nodes[n] and "lon" in G.nodes[n]
])

for metric, label in [
    ("In_Degree",   "Background vs Triggered (In-Degree)"),
    ("Out_Degree",  "Aftershock Productivity (Out-Degree)"),
    ("Degree",      "Most Productive Triggers"),
    ("PageRank",    "Long-Chain Influencers"),
    ("Harmonic",    "Topological Reach (Harmonic)"),
    ("Betweenness", "Sequence Bottlenecks"),
    ("HITS_Hub",    "Mainshock Hubs"),
    ("HITS_Auth",   "Aftershock Attractors"),
    ("Clustering",  "Multi-Path Events (Clustering)"),
    ("Triangles",   "Feedback Loops (Triangles)"),
]:
    print(f"\n--- Top 5: {label} ({metric}) ---")
    display(df_centrality.nlargest(5, metric)[
        ["cell_id", "lat", "lon", "depth_km", metric]])

plot_centrality_correlation(df_centrality, "BP Italy")
plot_top_n_cells(df_centrality, top_n=10, title="BP Italy")

## Centrality Geo Maps

Geographic projection of the top-10 events per centrality metric on the Italian
peninsula.  Two complementary views are produced:

1. **Per-metric interactive map** — 7 separate panels (one per centrality),
each showing the top-10 events coloured by their rank.
2. **Overlap map** — a single map where each event is coloured by how many
metrics it appears in simultaneously (warm = multi-metric dominance).

**Seismological interpretation:** events appearing in many metrics simultaneously
are structurally indispensable mainshocks — high productivity (degree/HITS),
long-chain influence (PageRank/Katz), and topological centrality (betweenness)
all converge on the same events.  These are the "pillars" of the aftershock
hierarchy: L'Aquila 2009, Amatrice–Norcia 2016, Irpinia 1980, and the
1997–1998 Umbria-Marche sequence.

**Expected output:** warm-coloured overlap concentrations in Central Apennines
and Southern Apennines.  If a single isolated event dominates all metrics,
it is likely the chronologically first event (the root of the spanning tree),
which has the highest PageRank by virtue of being the common ancestor of all others.

In [ ]:
plot_geo_top_n_interactive(
    df_centrality, top_n=10,
    title="BP Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)
plot_geo_centrality_overlap(
    df_centrality, top_n=10,
    title="BP Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)

## Bianconi-Barabasi Fitness

The Bianconi–Barabási (2001) fitness model generalises preferential attachment
by assigning each node an intrinsic fitness $\eta_i$, so that the probability
of attachment is $\pi_i \propto \eta_i k_i$.  Under this rule the degree grows as

$$k_i(t) \approx m\!\left(\frac{t}{t_i}\right)^{\beta_i},\qquad
\beta_i = \frac{\eta_i}{C}$$

The empirical fitness estimator $\hat{\beta}_i = \ln k_i(T) / \ln(T/t_i)$
isolates intrinsic seismogenic potential from the first-mover advantage
of early events: a high-$\hat{\beta}$ event accumulated connections
rapidly relative to its age in the catalog.

For the BP forest, $t_i$ is the actual earthquake time, $k_i(T)$ is the
final in-degree + out-degree of event $i$, and $T$ is the catalog duration.

Three theoretical regimes are compared against the observed $\rho(\hat{\beta})$:
- **Equal fitness (BA):** all events share the same $\eta$; $\hat{\beta} \approx 0.5$ spike.
- **Uniform fitness:** $\eta_i \sim U[0,1]$; $\hat{\beta}$ flat on $[0,\,\gamma-1]$.
- **Bose-Einstein condensation:** one event captures a finite fraction of
all edges — a single dominant mainshock (e.g. L'Aquila 2009 or Amatrice 2016).

*References:* Bianconi G. & Barabási A.-L. (2001). *EPL* 54, 436–442;
*PRL* 86, 5632–5635.

In [ ]:
print("Computing Bianconi-Barabasi fitness (BP events)…")
df_bb = compute_bb_fitness_events(G, df_net)
print(f"  {len(df_bb)} events with valid β̂; β range [{df_bb['fitness_beta'].min():.3f}, {df_bb['fitness_beta'].max():.3f}]")
plot_bb_fitness(df_bb, title="BP Italy")
plot_bb_fitness_theory(df_bb, gamma=gamma_bp, title="BP Italy")
plot_bb_fitness_geo(
    df_bb,
    title="BP Italy",
    center_lat=41.9, center_lon=12.5, zoom=0,
    bounds=_IT_BOUNDS, height=MAP_HEIGHT, width=MAP_WIDTH,
)

## Community Detection — Full Suite

Seven community-detection algorithms are applied to the **undirected** BP forest
(self-loops removed before conversion).  Modularity

$$Q = \frac{1}{2m}\sum_{ij}\left[A_{ij} - \frac{k_i k_j}{2m}\right]\delta(c_i,c_j)$$

(Newman & Girvan 2004) is the primary optimisation target for graph-based methods,
but density- and affiliation-based methods operate on complementary representations.

Unlike the Abe-Suzuki cell graph (where communities = geographic bins of frequently
co-visited spatial cells), BP communities reflect genuine triggering kinship: events
in the same community share a common seismogenic ancestor and are topologically close
in the aftershock tree.  A well-resolved community corresponds to a single aftershock
sequence or cluster of sequences that share the same fault segment.

**Graph-based methods**
- **Louvain** (Blondel et al. 2008): greedy modularity maximisation; fast and
scalable but subject to resolution limit and stochastic variation.
- **Consensus Louvain** (Lancichinetti & Fortunato 2012): 100-run ensemble average
over co-assignment frequencies; the most reproducible partition.
- **Spectral** (Von Luxburg 2007): Fiedler-vector bipartition applied recursively;
restricted to the giant component to avoid a rank-deficient disconnected Laplacian.
- **InfoMap** (directed; Rosvall & Bergstrom 2008): minimises the description length
of a random walk on the directed forest; often finds finer-grained communities than
modularity-based methods by respecting parent→child flow direction.

**Density-based methods**
- **HDBSCAN-Spectral** (Campello et al. 2013): applies HDBSCAN to the
$k$-dimensional spectral embedding of the normalised graph Laplacian.
The number of communities is determined entirely by the data — no $k$
pre-specification is required.  Points in low-density embedding regions are
labelled as noise and excluded from communities.
- **HDBSCAN-Geographic**: same HDBSCAN algorithm applied directly to projected
$(x, y)$ node coordinates in kilometres (EPSG:32632); no graph structure is used.
Comparing HDBSCAN-Spectral with HDBSCAN-Geographic quantifies how much of the
detected community structure is attributable to geographic proximity alone versus
the topology of the BP triggering tree.

**Overlapping-community method**
- **BigCLAM** (Yang & Leskovec 2013): solves for an $N \times k$ affiliation matrix
$F$ via non-negative matrix factorisation on the adjacency structure; the hard
partition is recovered by $\arg\max_c F_{ic}$.  In the BP context, events near
fault intersections may hold genuine affiliation in two communities
(e.g., the tail of one aftershock sequence and the onset of a triggered sequence
on an adjacent fault).

**Partition quality scoring**
All seven partitions are evaluated on 9 quality metrics — modularity, conductance,
coverage, Ncut, map equation, DC-SBM log-likelihood, Surprise, geographic
compactness, and depth coherence — via `compare_partitions`.

Pairwise NMI quantifies partition agreement across methods.  High NMI (> 0.6)
implies algorithmically robust community structure; low NMI (< 0.3) implies the
tree is too path-like for a clearly modular partition.

*References:* Newman & Girvan (2004) *Phys. Rev. E* 69, 026113;
Blondel et al. (2008) *J. Stat. Mech.* P10008;
Rosvall & Bergstrom (2008) *PNAS* 105, 1118;
Campello et al. (2013) *ECML-PKDD* 160–175;
Yang & Leskovec (2013) *ACM TKDD* 7(3), 1–42.

**Expected output:** Louvain finds 5–20 communities; InfoMap often finds 20–100+.
Communities should be geographically compact.  Pairwise NMI ≈ 0.2–0.6 (moderate;
tree modularity landscapes have many near-equivalent optima).
HDBSCAN-Geographic NMI with HDBSCAN-Spectral reveals whether BP triggering trees
are spatially co-located with geographic clusters.

In [ ]:
print("Building undirected BP graph for community detection…")
G_und_comm = G.to_undirected()
G_und_comm.remove_edges_from(nx.selfloop_edges(G_und_comm))

print("Louvain…")
community_map = run_louvain(G_und_comm, seed=42)
k_louvain     = len(set(community_map.values()))
print(f"  {k_louvain} communities")
plot_community_geo(
    G_und_comm, community_map,
    title="BP Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Louvain",
)

print("Consensus Louvain (100 runs)…")
community_map_consensus = run_consensus_louvain(G_und_comm, n_runs=100, seed=42)
print(f"  {len(set(community_map_consensus.values()))} communities")
plot_community_geo(
    G_und_comm, community_map_consensus,
    title="BP Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Consensus Louvain",
)

# Spectral on the giant component to avoid disconnected-Laplacian issues
print(f"Spectral clustering (k={min(k_louvain, 8)}, giant component)…")
_k_spec = min(k_louvain, 8)
community_map_spectral = run_spectral(G_und_giant, k=_k_spec, seed=42)
print(f"  {len(set(community_map_spectral.values()))} communities (giant component only)")
plot_community_geo(
    G_und_giant, community_map_spectral,
    title="BP Italy (giant component)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="Spectral",
)

print("InfoMap (directed)…")
community_map_infomap = run_infomap(G, directed=True, seed=42)
print(f"  {len(set(community_map_infomap.values()))} communities")
plot_community_geo(
    G, community_map_infomap,
    title="BP Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="InfoMap (directed)",
)

# NMI over nodes present in all partitions
_common_nodes = (set(community_map) & set(community_map_consensus)
                 & set(community_map_infomap))
partitions = {
    "Louvain":   {n: community_map[n]           for n in _common_nodes},
    "Consensus": {n: community_map_consensus[n] for n in _common_nodes},
    "InfoMap":   {n: community_map_infomap[n]   for n in _common_nodes},
}
print("Computing NMI across methods…")
nmi_df = compute_nmi_matrix(partitions)
display(nmi_df.round(3))
plot_nmi_heatmap(nmi_df, title="BP Italy")

print("HDBSCAN-Spectral…")
community_map_hdbscan_spec = run_hdbscan_spectral(G_und_giant, min_cluster_size=10, seed=42)
print(f"  {len(set(community_map_hdbscan_spec.values()))} clusters")
plot_community_geo(
    G_und_giant, community_map_hdbscan_spec,
    title="BP Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="HDBSCAN-Spectral",
)

print("HDBSCAN-Geographic…")
community_map_hdbscan_geo = run_hdbscan_geo(G_und_comm, min_cluster_size=10)
print(f"  {len(set(community_map_hdbscan_geo.values()))} clusters")
plot_community_geo(
    G_und_comm, community_map_hdbscan_geo,
    title="BP Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="HDBSCAN-Geographic",
)

print("BigCLAM…")
F_bigclam, community_map_bigclam = run_bigclam(
    G_und_comm, k=k_louvain, n_iter=100, lr=0.005, seed=42,
)
print(f"  {len(set(community_map_bigclam.values()))} non-empty communities")
plot_community_geo(
    G_und_comm, community_map_bigclam,
    title="BP Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="BigCLAM",
)

_hdbscan_spec_comm = {n: community_map_hdbscan_spec[n] for n in _common_nodes if n in community_map_hdbscan_spec}
_hdbscan_geo_comm  = {n: community_map_hdbscan_geo[n]  for n in _common_nodes if n in community_map_hdbscan_geo}
_bigclam_comm      = {n: community_map_bigclam[n]       for n in _common_nodes if n in community_map_bigclam}
_common_ext = set(_hdbscan_spec_comm) & set(_hdbscan_geo_comm) & set(_bigclam_comm)
partitions_ext = {
    **{k: {n: v[n] for n in _common_ext} for k, v in partitions.items()},
    "HDBSCAN-Spec": {n: _hdbscan_spec_comm[n] for n in _common_ext},
    "HDBSCAN-Geo":  {n: _hdbscan_geo_comm[n]  for n in _common_ext},
    "BigCLAM":      {n: _bigclam_comm[n]       for n in _common_ext},
}
nmi_ext = compute_nmi_matrix(partitions_ext)
display(nmi_ext.round(3))
plot_nmi_heatmap(nmi_ext, title="BP Italy — all methods")

scores_df = compare_partitions(G_und_comm, partitions_ext, cell_size_km=10.0)
display(scores_df.round(4))
plot_partition_scores(scores_df, title="BP Italy")

## Directed Community Detection

Directed modularity:

$$Q_d = \frac{1}{m}\sum_{ij}\left[A_{ij} - \frac{k_i^{\text{out}}k_j^{\text{in}}}{m}\right]\delta(c_i,c_j)$$

(Leicht & Newman 2008) captures the asymmetric flow structure of the triggering
forest.  A community under $Q_d$ is a set of events that preferentially trigger
*each other* within the group — seismologically, a self-contained aftershock
cluster where triggering flows are internally concentrated.

The Leiden algorithm (a rigorous Louvain variant) is used for directed modularity
maximisation.

**NMI vs undirected Louvain:** high NMI means the triggering direction does not
reorganise the communities (symmetric fault zones, similar forward and backward
coupling).  Low NMI reveals directional cascades — mainshocks that trigger
aftershocks in a neighbouring zone but never receive triggers from it, splitting
communities along the propagation direction.

**Expected output:** directed Louvain typically finds a similar number of
communities to undirected Louvain (± 20%).  NMI between directed and undirected
≈ 0.3–0.6.  Community geo maps show whether directional asymmetries reorganise
which events cluster together.

In [ ]:
print("Running directed Louvain (Leiden) on BP network…")
community_map_directed = run_directed_louvain(G, seed=42)
k_directed = len(set(community_map_directed.values()))
print(f"  {k_directed} directed communities  (vs {k_louvain} undirected)")

plot_community_geo(
    G, community_map_directed,
    title="BP Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Directed Louvain (Leiden)",
)

_common_dir = _common_nodes & set(community_map_directed)
partitions_full = {
    **{k: {n: v[n] for n in _common_dir} for k, v in partitions.items()},
    "Directed Louvain": {n: community_map_directed[n] for n in _common_dir},
}
nmi_full = compute_nmi_matrix(partitions_full)
display(nmi_full.round(3))
plot_nmi_heatmap(nmi_full, title="BP Italy — directed vs undirected")

## Community Markov Flow

The Louvain partition coarse-grains the BP forest into a $K \times K$ flow
matrix $F_{ij} = \sum_{u \in C_i,\, v \in C_j} w_{uv}$, where $w_{uv}$ is the
BP edge weight.  Row-normalisation gives the Markov transition matrix $T_{ij}$:
the probability that the next triggered event after a community-$i$ earthquake
falls into community $j$.

**Physical interpretation:** $T_{ij}$ measures inter-zone stress transfer.
High off-diagonal $T_{ij}$ identifies fault segments that consistently pass
seismic stress across tectonic province boundaries.  The stationary distribution
$\boldsymbol{\pi}$ (eigenvector of $T$ for eigenvalue 1) gives the long-run
fraction of seismic activity absorbed by each community — the dominant zone
in the Markov chain corresponds to the most energetically active fault system.

**Shannon outflow entropy** $H_i = -\sum_j T_{ij} \ln T_{ij}$ measures how
diffusely a community distributes its triggered events.  High $H_i$ means
community $i$ triggers events spread across many other communities (a broadly
coupled zone); low $H_i$ means triggering is concentrated locally
(self-contained aftershock cluster).

*Reference:* Rosvall, M., & Bergstrom, C. T. (2008). *PNAS*, 105(4), 1118–1123.

**Expected output:** $K$ = number of Louvain communities; mean self-retention
$T_{ii} \approx 0.7$–$0.95$; mean outflow entropy ≈ 1–3 bits.  The dominant
attractor (highest $\pi$) corresponds to the community hosting the densest /
most energetic seismicity.

In [ ]:
from src.community_flow import (
    build_community_flow_matrix,
    compute_markov_chain,
    compute_stationary_distribution,
    community_flow_stats,
    plot_flow_heatmap,
    plot_flow_entropy,
    plot_stationary_distribution,
)

print("Building community flow matrix (Louvain, BP network)…")
flow_count_df = build_community_flow_matrix(G, community_map)
T_markov  = compute_markov_chain(flow_count_df)
stat_dist = compute_stationary_distribution(T_markov)
df_flow   = community_flow_stats(T_markov, flow_count_df, stat_dist)

K_comm = T_markov.shape[0]
print(f"Markov chain: {K_comm} community states")
print(f"Mean self-retention:  {df_flow['self_retention'].mean():.3f}")
print(f"Mean outflow entropy: {df_flow['entropy'].mean():.3f} bits "
      f"(max = log₂({K_comm}) = {np.log2(K_comm):.3f})")
print(f"Dominant attractor:   Community {df_flow.iloc[0]['community']} "
      f"(π = {df_flow.iloc[0]['stationary']:.4f})")
display(df_flow)

plot_flow_heatmap(T_markov, flow_count_df, title="BP Italy")
plot_flow_entropy(df_flow, K=K_comm, title="BP Italy")
plot_stationary_distribution(df_flow, title="BP Italy")

out_path = RESULTS_DIR / "data" / "italy_bp_community_flow.csv"
df_flow.to_csv(out_path, index=False)
print(f"Community flow stats saved → {out_path}")

## Assortativity

Degree–degree and attribute–attribute correlations (Newman 2003) quantify
whether the BP forest preferentially links events of similar properties.

$$r = \frac{\sum_{ij}j k_i k_j (e_{ij} - q_i q_j)}{\sigma_q^2}$$

where $e_{ij}$ is the fraction of edges linking degree-$i$ to degree-$j$ nodes
and $q_i = \sum_j e_{ij}$.

**Magnitude assortativity** $r < 0$ (disassortative) is physically expected:
mainshocks (high $m_i$) preferentially trigger aftershocks (lower $m_j$) —
Båth's law expressed as a network property.  Values of $r \approx -0.1$ to
$-0.3$ are typical.

**Depth assortativity** $r \approx 0$: crustal events trigger both crustal and
sub-crustal aftershocks in Italy, so depth correlation is weak.

**Degree assortativity** $r < 0$ (disassortative, hub-and-spoke): high-degree
mainshocks connect to many low-degree aftershock leaves.

*Reference:* Newman, M. E. J. (2003). Mixing patterns in networks.
*Physical Review E*, 67(2), 026126.

**Expected output:** scatter matrix with 3 panels (degree × degree,
magnitude × magnitude, depth × depth).  Each panel shows a 2D histogram
with a regression line.  Negative slope in the magnitude panel confirms
Båth's law; near-zero slope in the depth panel confirms cross-depth coupling.

In [ ]:
print("Computing assortativity (BP node attributes already attached)…")
df_assort = compute_assortativity(G)
display(df_assort)
plot_assortativity(G, title="BP Italy")

print("Average nearest-neighbour degree k_nn(k):")
plot_knn(G, title="BP Italy", gamma=gamma_bp)

print("Rich-club coefficient:")
plot_rich_club(G, title="BP Italy")

## Export Results

Three output files are written:

1. **CSV** (`italy_bp_network_metrics.csv`) — one row per event with centrality
values, geographic coordinates, depth, and Louvain community assignment.
Used for statistical analysis and report tables.
2. **Pickle** (`italy_bp.pkl`) — the full NetworkX DiGraph.  Reload in
≈ 5 seconds for the comparison notebook (`USE_CACHED_NETWORKS = True`).
3. **GEXF** (`italy_bp.gexf`) — Gephi-compatible XML for visual graph exploration
and force-directed layout.

**Expected output:** CSV ≈ 215,000 rows × 12 columns; pickle ≈ 200–400 MB;
GEXF ≈ 100–300 MB.  All three paths are printed on completion.

In [ ]:
df_comm = pd.DataFrame(
    [{"cell_id": n, "community": community_map[n]} for n in community_map]
)
df_final = df_centrality.merge(
    df_comm[["cell_id", "community"]], on="cell_id", how="left"
)
out_path = RESULTS_DIR / "data" / "italy_bp_network_metrics.csv"
df_final.to_csv(out_path, index=False)
print(f"Results saved → {out_path}  ({len(df_final):,} rows)")

pkl_path = RESULTS_DIR / "cache" / "italy_bp.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump(G, f)
print(f"Network cached → {pkl_path}")

gexf_path = RESULTS_DIR / "gephi" / "italy_bp.gexf"
nx.write_gexf(G, gexf_path)
print(f"Gephi export → {gexf_path}")